# Introdução prática ao Gemini API — Free Tier

Este notebook mostra o básico para interagir com uma IA usando a **Gemini API** em Python.

Você vai aprender a:

1. instalar a biblioteca oficial `google-genai`;
2. configurar a chave da API;
3. enviar uma pergunta simples para o Gemini;
4. usar instrução de sistema;
5. controlar criatividade com `temperature`;
6. criar um chat simples com histórico;
7. gerar respostas estruturadas em JSON.

> Material pensado para Google Colab e para uso didático em aula.

## 1. Instalação da biblioteca

A biblioteca `google-genai` permite acessar os modelos Gemini usando Python.

No Google Colab, execute a célula abaixo para instalar o pacote.

In [ ]:
!pip install google-genai

## 2. Configuração da chave da API

Para usar a Gemini API, você precisa de uma chave de API.

Ela pode ser criada no Google AI Studio.

Nesta célula, usamos `getpass` para esconder a chave enquanto ela é digitada.

In [3]:
import os
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Cole sua GEMINI_API_KEY: ")

Cole sua GEMINI_API_KEY:  ········


## 3. Criando o cliente Gemini

O cliente é o objeto responsável por se comunicar com a API do Gemini.

In [4]:
from google import genai

cliente = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## 4. Escolhendo o modelo

Para aulas e testes rápidos no free tier, modelos da família **Flash** costumam ser uma boa escolha.

Caso o modelo abaixo não esteja disponível na sua conta, troque por outro modelo Flash disponível no Google AI Studio.

In [5]:
MODELO = "gemini-2.5-flash"

## 5. Primeira pergunta para a IA

Agora vamos enviar uma pergunta simples para testar se a conexão está funcionando.

In [7]:
resposta = cliente.models.generate_content(
    model=MODELO,
    contents="O que eu acabei de te perguntar?"
)

print(resposta.text)

Você acabou de me perguntar: "O que eu acabei de te perguntar?"


## 6. Criando uma função para perguntar ao Gemini

Agora vamos organizar o código em uma função reutilizável.

Assim, sempre que quisermos fazer uma pergunta, basta chamar `perguntar_ao_gemini()`.

In [ ]:
def perguntar_ao_gemini(pergunta: str) -> str:
    """
    Envia uma pergunta para o Gemini e retorna a resposta em texto.
    """

    resposta = cliente.models.generate_content(
        model=MODELO,
        contents=pergunta
    )

    return resposta.text

In [ ]:
resposta = perguntar_ao_gemini("Me dê 3 exemplos de uso de IA no dia a dia.")
print(resposta)

## 7. Usando uma instrução de sistema

A **instrução de sistema** define como a IA deve se comportar.

Exemplo: responder em português, ser objetiva, agir como professora, explicar passo a passo etc.

In [ ]:
INSTRUCAO_SISTEMA = (
    "Você é um professor de tecnologia. "
    "Explique os assuntos em português do Brasil, "
    "com linguagem simples, exemplos práticos e sem respostas longas demais."
)

resposta = cliente.models.generate_content(
    model=MODELO,
    contents="Explique o que é uma API para um aluno iniciante.",
    config={
        "system_instruction": INSTRUCAO_SISTEMA
    }
)

print(resposta.text)

## 8. Controlando a criatividade com `temperature`

O parâmetro `temperature` controla o nível de criatividade da resposta.

- Valores baixos, como `0.1` ou `0.2`, deixam a resposta mais direta e previsível.
- Valores médios, como `0.5`, equilibram clareza e criatividade.
- Valores altos, como `0.9`, deixam a resposta mais criativa, mas menos previsível.

In [ ]:
pergunta = "Crie uma analogia simples para explicar o que é machine learning."

resposta = cliente.models.generate_content(
    model=MODELO,
    contents=pergunta,
    config={
        "temperature": 0.7,
        "system_instruction": INSTRUCAO_SISTEMA
    }
)

print(resposta.text)

## 9. Limitando o tamanho da resposta

O parâmetro `max_output_tokens` ajuda a limitar o tamanho da resposta.

Isso é útil para economizar uso da API e evitar respostas muito longas.

In [ ]:
resposta = cliente.models.generate_content(
    model=MODELO,
    contents="Explique o que é Python em uma resposta curta.",
    config={
        "system_instruction": INSTRUCAO_SISTEMA,
        "max_output_tokens": 120
    }
)

print(resposta.text)

## 10. Chat simples com histórico

Um chatbot precisa lembrar das mensagens anteriores.

Para isso, vamos guardar o histórico em uma lista.

Cada mensagem será armazenada com um papel:

- `user`: mensagem enviada pelo usuário;
- `model`: resposta gerada pela IA.

In [ ]:
historico = []

def conversar(mensagem_usuario: str) -> str:
    """
    Envia uma mensagem para o Gemini usando o histórico da conversa.
    """

    historico.append({
        "role": "user",
        "parts": [{"text": mensagem_usuario}]
    })

    resposta = cliente.models.generate_content(
        model=MODELO,
        contents=historico,
        config={
            "system_instruction": INSTRUCAO_SISTEMA,
            "temperature": 0.4
        }
    )

    texto_resposta = resposta.text

    historico.append({
        "role": "model",
        "parts": [{"text": texto_resposta}]
    })

    return texto_resposta

In [ ]:
print(conversar("Meu nome é Ana e estou aprendendo Python."))

In [ ]:
print(conversar("Qual é o meu nome e o que eu estou aprendendo?"))

## 11. Limpando o histórico

Se quisermos começar uma conversa nova, basta limpar a lista `historico`.

In [ ]:
historico.clear()
print("Histórico limpo!")

## 12. Pedindo resposta em formato JSON

Às vezes queremos que a IA responda em um formato estruturado.

Isso é útil quando queremos usar a resposta dentro de outro programa.

In [ ]:
prompt = """
Crie um plano de estudos simples sobre Python básico.

Responda somente em JSON válido, seguindo este formato:
{
  "tema": "",
  "duracao_dias": 0,
  "topicos": [],
  "projeto_final": ""
}
"""

resposta = cliente.models.generate_content(
    model=MODELO,
    contents=prompt,
    config={
        "temperature": 0.2,
        "system_instruction": "Você responde somente em JSON válido, sem texto antes ou depois."
    }
)

print(resposta.text)

## 13. Exercícios para os alunos

Faça as atividades abaixo:

1. Altere o prompt da primeira pergunta para pedir uma explicação sobre banco de dados.
2. Mude a `temperature` para `0.1` e depois para `0.9`. Compare as respostas.
3. Crie uma instrução de sistema dizendo que a IA deve responder como um professor do SENAI.
4. Use o chat com histórico para fazer três perguntas conectadas sobre Python.
5. Peça para a IA gerar uma lista de exercícios em JSON.

## 14. Desafio final

Crie uma função chamada `gerar_exercicios()` que receba um tema e retorne 5 exercícios sobre esse tema.

Exemplo esperado:

```python
gerar_exercicios("listas em Python")
```

A função deve pedir para o Gemini gerar exercícios com:

- enunciado;
- nível de dificuldade;
- resposta esperada.

In [ ]:
def gerar_exercicios(tema: str) -> str:
    prompt = f"""
    Crie 5 exercícios sobre o tema: {tema}.

    Para cada exercício, informe:
    - enunciado;
    - nível de dificuldade;
    - resposta esperada.

    Responda em português do Brasil.
    """

    resposta = cliente.models.generate_content(
        model=MODELO,
        contents=prompt,
        config={
            "system_instruction": INSTRUCAO_SISTEMA,
            "temperature": 0.4
        }
    )

    return resposta.text

In [ ]:
print(gerar_exercicios("listas em Python"))

## Fechamento

Neste notebook você viu o básico para usar o Gemini com Python:

- instalação da biblioteca;
- configuração da chave;
- envio de prompts;
- instruções de sistema;
- parâmetros básicos;
- histórico de conversa;
- resposta estruturada;
- criação de exercícios com IA.

A partir daqui, você pode evoluir para projetos com interface web usando Streamlit ou Gradio.

# Chatbot com Interface usando Gradio

In [ ]:
!pip -q install google-genai gradio

In [4]:
import os
import getpass
os.environ["GEMINI_API_KEY"] = getpass.getpass("Cole sua GEMINI_API_KEY: ")

Cole sua GEMINI_API_KEY:  ········


In [3]:
import os
import getpass
from google import genai
cliente = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
#LANGCHAIN -> comentário
MODELO = "gemini-2.5-flash"
SYSTEM_INSTRUCTION ="""
Você é um chatbot educado e objetivo.
Responda em Português do Brasil, com clareza e passos práticos.
Se faltar informação, faça 1 pergunta curta antes de responder.

"""

def chat_com_historico(mensagem_usuario, historico):
    conteudo = []
    for usuario, agente in historico:
        if usuario:
            conteudo.append({"role":"user", "content":[{"text":usuario}]})
        if agente:
            conteudo.append({"role":"model", "content":[{"text":agente}]})
    #Mensagem atual
    conteudo.append({"role": "user", "content":[{"text":mensagem_usuario}]})
    
    resposta = cliente.models.generate_content(model = MODELO, contents=conteudo,
                                              config = {"system_instruction": SYSTEM_INSTRUCTION,
                                                        "temperature": 0.4,},)
    return resposta.text
    


KeyError: 'GEMINI_API_KEY'

In [26]:

import gradio as gr

with gr.Blocks(title="Super Chatbot Módulo 3") as demo:
    gr.Markdown("## Chatbot com histórico!")

    chat = gr.Chatbot(height=500)
    msg = gr.Textbox(placeholder="Digite sua pergunta: ",show_label=False)
    clear = gr.Button("Limpar")

    def responder(mensagem_usuario, historico):
        resp = chat_com_historico(mensagem_usuario, historico)
        historico = historico + [(mensagem_usuario, resp)]
        return "",historico

    msg.submit(responder, [msg, chat], [msg,chat])
    clear.click(lambda: [], None, chat)

demo.launch()
    

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "C:\Users\prof01\AppData\Roaming\Python\Python313\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "C:\Users\prof01\AppData\Roaming\Python\Python313\site-packages\gradio\route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "C:\Users\prof01\AppData\Roaming\Python\Python313\site-packages\gradio\blocks.py", line 2195, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "C:\Users\prof01\AppData\Roaming\Python\Python313\site-packages\gradio\blocks.py", line 1652, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^